# 04 · Axis-wise W1 σ — Frenet 잔차 비대칭에 비례한 가중 (EXP #17, LB 0.6906)

직전 = #15 V3 multi-seed(LB 0.6888). W1 가중의 σ가 세 축 공통(3.5mm)이었다.

**가정**: Frenet 잔차 std가 축별로 비대칭(L:N:B = 14.74:11.93:9.20mm ≈ 1:0.81:0.62)이다. Cartesian(1:0.96:0.68)에선 거의 대칭이라 axis-wise 가중이 D5에서 무효였지만, **비대칭이 큰 Frenet 축에선 std에 비례한 축별 σ**가 boundary 가중을 의미있게 만든다.

**실험**: σ를 (σ_L, σ_N, σ_B)로 분리, 비례 정도가 다른 4 config(V3_ref 대칭 / C1 1:1 비례 / C2 super / C3 sub)를 3 seed로 비교. V3_ref가 #15 재현(control)인지 sanity 후 3-gate.

**결과**: **C1(4.3/3.5/2.7mm) interior optimum** 채택 → **LB 0.6906 (+0.0018)**, minority +0.0028 selective. axis-wise 가중의 효과는 **잔차 std 비대칭 정도가 결정 인자**임을 확인(RESEARCH §6 W5).

## 셀 0 — imports + 상수

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

DT, DT_PRED = 0.04, 0.08
N_FOLDS = 5
MINORITY_THRESHOLD = 5.0
HIT_RADIUS = 0.01

SEEDS = [42, 7, 123]
BEST_MAX_W = 2.5

# Configs: axis-wise σ (L, N, B 순)
CONFIGS = {
    'V3_ref': (0.0035, 0.0035, 0.0035),  # control (V3 multi-seed reference)
    'C1':     (0.0043, 0.0035, 0.0027),  # 1:1 std proportional (anchor)
    'C2':     (0.0050, 0.0035, 0.0020),  # super-proportional (~1.4x)
    'C3':     (0.0040, 0.0035, 0.0030),  # sub-proportional (~0.6x)
}

# V3 multi-seed reference (EXP #15, plan §13.5)
V3_REF_OVERALL = 0.6615
V3_REF_OVERALL_STD = 0.0007
V3_REF_MAJOR = 0.7292
V3_REF_MINOR = 0.2995
V3_REF_P1 = 0.6607

os.makedirs('results', exist_ok=True)
print(f'SEEDS={SEEDS}, configs={list(CONFIGS.keys())}')
for cn, sg in CONFIGS.items():
    print(f'  {cn}: σ_L={sg[0]*1000:.1f}mm, σ_N={sg[1]*1000:.1f}mm, σ_B={sg[2]*1000:.1f}mm')
print(f'V3 reference: overall {V3_REF_OVERALL}±{V3_REF_OVERALL_STD}, P1 {V3_REF_P1}')

## 셀 1 — helper (T 노트북 동일 + axis-wise weight 추가)

In [ ]:
def _norm(x):
    return np.linalg.norm(x, axis=-1)


def physics_baseline(traj):
    p0, p_m40 = traj[:, -1, :], traj[:, -2, :]
    v_last = (p0 - p_m40) / DT
    return (p0 + v_last * DT_PRED).astype(np.float32)


def hit_rate(pred, label, r=HIT_RADIUS):
    return float((np.linalg.norm(pred - label, axis=1) < r).mean())


def w_gaussian_one(e, sigma, max_w, r=HIT_RADIUS):
    return np.clip(1.0 + (max_w - 1.0) * np.exp(-(e - r)**2 / (2 * sigma**2)),
                   0.5, max_w)


def w_axis_3(e_cart, sigmas, max_w):
    """Return (N, 3) axis-wise weights. sigmas=(σ_L, σ_N, σ_B)."""
    return np.stack([
        w_gaussian_one(e_cart, sigmas[0], max_w),
        w_gaussian_one(e_cart, sigmas[1], max_w),
        w_gaussian_one(e_cart, sigmas[2], max_w),
    ], axis=1).astype(np.float32)


def make_splits(minority_int, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in skf.split(np.arange(len(minority_int)), minority_int)]


lgbm_params = dict(
    objective='regression_l1', metric='mae',
    learning_rate=0.05, num_leaves=31, min_data_in_leaf=5,
    max_bin=511, n_estimators=500, random_state=42, verbose=-1,
)
print('helper 정의 완료')

## 셀 2 — Drive mount + 데이터 로드 (캐시 우선)

In [ ]:
CACHE_TRAJ_TR = 'traj_train.npy'
CACHE_Y_TR    = 'y_train.npy'
CACHE_TRAJ_TS = 'traj_test.npy'

DATA_DIR = None
if not (os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR) and os.path.exists(CACHE_TRAJ_TS)):
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/open.zip'
    !unzip -q -o "{ZIP_PATH}" -d /content/

def _resolve_data_dir():
    for cand in ['/content/open', '/content']:
        if os.path.exists(f'{cand}/train_labels.csv'):
            return cand
    return None

def _resolve_sample_sub(data_dir):
    for p in [f'{data_dir}/sample_submission.csv',
              f'{data_dir}/submission/sample_submission.csv']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'sample_submission.csv not found under {data_dir}')

DATA_DIR = _resolve_data_dir()
if DATA_DIR is not None:
    print(f'DATA_DIR = {DATA_DIR}')
else:
    print('cache 존재 — DATA_DIR resolve skip')

if os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR):
    traj_train = np.load(CACHE_TRAJ_TR)
    y_train    = np.load(CACHE_Y_TR)
else:
    labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
    train_ids = labels['id'].tolist()
    y_train = labels[['x','y','z']].values.astype(np.float32)
    traj_train = np.empty((len(train_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(train_ids):
        df = pd.read_csv(f'{DATA_DIR}/train/{tid}.csv')
        traj_train[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TR, traj_train)
    np.save(CACHE_Y_TR, y_train)

if DATA_DIR is None:
    DATA_DIR = _resolve_data_dir()
    if DATA_DIR is None:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
        !unzip -q -o /content/drive/MyDrive/open.zip -d /content/
        DATA_DIR = _resolve_data_dir()
        assert DATA_DIR is not None, 'DATA_DIR resolve fail after unzip'

sample_sub_path = _resolve_sample_sub(DATA_DIR)
print(f'sample_submission: {sample_sub_path}')
sample_sub = pd.read_csv(sample_sub_path)
test_ids = sample_sub['id'].tolist()

if os.path.exists(CACHE_TRAJ_TS):
    traj_test = np.load(CACHE_TRAJ_TS)
else:
    traj_test = np.empty((len(test_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(test_ids):
        df = pd.read_csv(f'{DATA_DIR}/test/{tid}.csv')
        traj_test[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TS, traj_test)

assert traj_train.shape == (10000, 11, 3) and traj_test.shape == (10000, 11, 3)
print(f'train {traj_train.shape}, test {traj_test.shape}')

## 셀 3 — base + Frenet frame + kinematics

In [ ]:
def build_frenet_batch(v_last, a_sm, fb_thresh=1e-6):
    vn = np.linalg.norm(v_last, axis=1, keepdims=True)
    eL = v_last / (vn + 1e-9)
    a_perp = a_sm - (a_sm * eL).sum(1, keepdims=True) * eL
    apn = np.linalg.norm(a_perp, axis=1, keepdims=True)
    eN_p = a_perp / (apn + 1e-9)
    z_up = np.array([0., 0., 1.]); y_up = np.array([0., 1., 0.])
    n1 = np.linalg.norm(np.cross(eL, z_up), axis=1, keepdims=True)
    eN_fb1 = np.cross(eL, z_up) / (n1 + 1e-9)
    eN_fb2 = np.cross(eL, y_up); n2 = np.linalg.norm(eN_fb2, axis=1, keepdims=True)
    eN_fb  = np.where(n1 < 1e-6, eN_fb2 / (n2 + 1e-9), eN_fb1)
    eN = np.where(apn < fb_thresh, eN_fb, eN_p)
    eB = np.cross(eL, eN)
    eB = eB / (np.linalg.norm(eB, axis=1, keepdims=True) + 1e-9)
    return eL.astype(np.float32), eN.astype(np.float32), eB.astype(np.float32)


def compute_kinematics(traj):
    v = (traj[:, 1:, :] - traj[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last = v[:, -1, :]
    a_last = a[:, -1, :]
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    w = np.array([1, 2, 3]) / 6
    a_sm = (a[:, -3:, :] * w[None, :, None]).sum(axis=1)
    return v_last.astype(np.float32), a_last.astype(np.float32), jerk_last.astype(np.float32), a_sm.astype(np.float32)


base_train = physics_baseline(traj_train)
base_test  = physics_baseline(traj_test)
residual_cart_train = (y_train - base_train).astype(np.float32)
vl_tr, al_tr, jl_tr, asm_tr = compute_kinematics(traj_train)
vl_ts, al_ts, jl_ts, asm_ts = compute_kinematics(traj_test)
eL_tr, eN_tr, eB_tr = build_frenet_batch(vl_tr, asm_tr)
eL_ts, eN_ts, eB_ts = build_frenet_batch(vl_ts, asm_ts)

r_L = (residual_cart_train * eL_tr).sum(1)
r_N = (residual_cart_train * eN_tr).sum(1)
r_B = (residual_cart_train * eB_tr).sum(1)
rec = r_L[:, None]*eL_tr + r_N[:, None]*eN_tr + r_B[:, None]*eB_tr
inv_err = np.abs(rec - residual_cart_train).max()
assert inv_err < 1e-5
residual_fren_train = np.stack([r_L, r_N, r_B], axis=1).astype(np.float32)
print(f'frame inverse max err: {inv_err:.2e}')
print(f'residual_fren std L:N:B = {residual_fren_train.std(axis=0)*1000} mm')

acc_norm_last_tr = _norm(al_tr)
minority_mask_tr = acc_norm_last_tr >= MINORITY_THRESHOLD
print(f'minority: {minority_mask_tr.sum()}/{len(minority_mask_tr)}')

## 셀 4 — V3 26 feat (19 scalar + 7 Frenet projection)

In [ ]:
def build_v3_31(traj):
    p = traj
    v = (p[:, 1:, :] - p[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last, a_last = v[:, -1, :], a[:, -1, :]
    speed_last, acc_norm_last = _norm(v_last), _norm(a_last)
    w = np.array([1, 2, 3]) / 6
    a3 = a[:, -3:, :]
    a_w3 = (a3 * w[None, :, None]).sum(axis=1)
    acc_norm_w3 = _norm(a_w3)
    v_std, a_std = v.std(axis=1), a.std(axis=1)
    seg = _norm(p[:, 1:, :] - p[:, :-1, :])
    path_eff = _norm(p[:, -1, :] - p[:, 0, :]) / (seg.sum(1) + 1e-9)
    p0 = p[:, -1, :]
    distance_r = _norm(p0)
    radial_v = (p0 * v_last).sum(1) / (distance_r + 1e-9)
    v_n = v / (_norm(v)[..., None] + 1e-9)
    cos_consec = (v_n[:, 1:, :] * v_n[:, :-1, :]).sum(-1).clip(-1, 1)
    turn = np.arccos(cos_consec)
    turn_mean, cos_turn_last = turn.mean(1), cos_consec[:, -1]
    va_dot = (v_last * a_last).sum(1) / (speed_last * acc_norm_last + 1e-9)
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    v_a_dot = (v_last * a_last).sum(1)
    v_cross_a = np.cross(v_last, a_last)
    a_tang_last = v_a_dot / (speed_last + 1e-9)
    a_cent_last = _norm(v_cross_a) / (speed_last + 1e-9)
    speed_seq = _norm(v)
    speed_diff_half = speed_seq[:, 5:].mean(1) - speed_seq[:, :5].mean(1)
    turn_mean_half_diff = turn[:, 4:].mean(1) - turn[:, :4].mean(1)
    return (
        np.column_stack([
            v_last, a_last, speed_last, acc_norm_last,
            a_w3, acc_norm_w3,
            v_std, a_std, path_eff,
            distance_r, radial_v,
            turn_mean, cos_turn_last, va_dot,
            jerk_last,
            a_tang_last, a_cent_last,
            speed_diff_half, turn_mean_half_diff,
        ]).astype(np.float32),
        a_last, jerk_last, a_w3,
    )


def fren_proj(a_last, jerk_last, a_w3, eL, eN, eB):
    a_N  = (a_last * eN).sum(1); a_B = (a_last * eB).sum(1)
    j_L  = (jerk_last * eL).sum(1); j_N = (jerk_last * eN).sum(1); j_B = (jerk_last * eB).sum(1)
    aw_L = (a_w3 * eL).sum(1); aw_N = (a_w3 * eN).sum(1)
    return np.column_stack([a_N, a_B, j_L, j_N, j_B, aw_L, aw_N]).astype(np.float32)


CART_DIRS_IDX = list(range(0, 6)) + list(range(8, 11)) + list(range(24, 27))
SCALAR_IDX = [i for i in range(31) if i not in CART_DIRS_IDX]
assert len(SCALAR_IDX) == 19

def build_v3_26(traj, eL, eN, eB):
    X31, a_last, jerk_last, a_w3 = build_v3_31(traj)
    F7 = fren_proj(a_last, jerk_last, a_w3, eL, eN, eB)
    return np.concatenate([X31[:, SCALAR_IDX], F7], axis=1)


X_tr = build_v3_26(traj_train, eL_tr, eN_tr, eB_tr)
X_ts = build_v3_26(traj_test,  eL_ts, eN_ts, eB_ts)
assert X_tr.shape == (10000, 26) and X_ts.shape == (10000, 26)
print(f'X_tr {X_tr.shape}, X_ts {X_ts.shape}')

## 셀 5 — run_one_config_seed (Phase 1 동일 + Phase 2 axis-wise)

Phase 1: no-weight (모든 config 동일, e_cart 시드만 다시 계산하되 V3와 동일해야 함).
Phase 2: axis ax에 대해 w_ax = w_gaussian(e_cart, σ_ax, max_w).

In [ ]:
def inverse_fren_to_cart(resid_fren, eL, eN, eB):
    return (resid_fren[:, 0:1] * eL + resid_fren[:, 1:2] * eN + resid_fren[:, 2:3] * eB)


def run_one_config_seed(config_name, sigmas, seed, max_w=BEST_MAX_W):
    """Phase 1 (no-weight) → Phase 2 (axis-wise W1). sigmas=(σ_L, σ_N, σ_B)."""
    folds = make_splits(minority_mask_tr.astype(int), seed=seed)

    # ── Phase 1: no-weight (Frenet target) ──
    oof_resid_p1 = np.full((10000, 3), np.nan, dtype=np.float32)
    for tr_idx, val_idx in folds:
        for ax in range(3):
            m = lgb.LGBMRegressor(**lgbm_params)
            m.fit(X_tr[tr_idx], residual_fren_train[tr_idx, ax],
                  eval_set=[(X_tr[val_idx], residual_fren_train[val_idx, ax])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            oof_resid_p1[val_idx, ax] = m.predict(X_tr[val_idx]).astype(np.float32)

    oof_cart_p1 = inverse_fren_to_cart(oof_resid_p1, eL_tr, eN_tr, eB_tr)
    pred_p1 = base_train + oof_cart_p1
    oof_e = np.linalg.norm(y_train - pred_p1, axis=1).astype(np.float32)
    HR_p1 = hit_rate(pred_p1, y_train)

    # ── Phase 2: axis-wise W1 ──
    w_axis_arr = w_axis_3(oof_e, sigmas, max_w)  # (10000, 3)
    oof_resid_p2 = np.full((10000, 3), np.nan, dtype=np.float32)
    test_resid_p2 = np.zeros((10000, 3), dtype=np.float32)
    for tr_idx, val_idx in folds:
        for ax in range(3):
            m = lgb.LGBMRegressor(**lgbm_params)
            m.fit(X_tr[tr_idx], residual_fren_train[tr_idx, ax],
                  sample_weight=w_axis_arr[tr_idx, ax],
                  eval_set=[(X_tr[val_idx], residual_fren_train[val_idx, ax])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            oof_resid_p2[val_idx, ax] = m.predict(X_tr[val_idx]).astype(np.float32)
            test_resid_p2[:, ax] += m.predict(X_ts).astype(np.float32) / N_FOLDS

    oof_cart_p2 = inverse_fren_to_cart(oof_resid_p2, eL_tr, eN_tr, eB_tr)
    test_cart_p2 = inverse_fren_to_cart(test_resid_p2, eL_ts, eN_ts, eB_ts)
    pred_p2 = base_train + oof_cart_p2
    HR_p2 = hit_rate(pred_p2, y_train)
    HR_p2_major = hit_rate(pred_p2[~minority_mask_tr], y_train[~minority_mask_tr])
    HR_p2_minor = hit_rate(pred_p2[ minority_mask_tr], y_train[ minority_mask_tr])

    return dict(
        config=config_name, sigmas=sigmas, seed=seed,
        HR_p1=HR_p1, HR_p2_overall=HR_p2, HR_p2_major=HR_p2_major, HR_p2_minor=HR_p2_minor,
        oof_resid_cart=oof_cart_p2, test_resid_cart=test_cart_p2,
    )

print('helper ready')

## 셀 6 — Multi-seed 학습 (4 configs × 3 seed = 12 runs)

360 model, ~22분.

In [ ]:
all_results = {}
test_preds = {c: [] for c in CONFIGS}
oof_preds  = {c: [] for c in CONFIGS}

for cname, sigmas in CONFIGS.items():
    for seed in SEEDS:
        t0 = time.time()
        r = run_one_config_seed(cname, sigmas, seed)
        all_results[(cname, seed)] = r
        test_preds[cname].append(r['test_resid_cart'])
        oof_preds[cname].append(r['oof_resid_cart'])
        np.save(f'results/t_ax_oof_{cname}_seed{seed}.npy', r['oof_resid_cart'])
        np.save(f'results/t_ax_test_{cname}_seed{seed}.npy', r['test_resid_cart'])
        print(f'  {cname} σ={tuple(round(s*1000,1) for s in sigmas)}mm seed={seed:3d}: '
              f'P1={r["HR_p1"]:.4f}  P2={r["HR_p2_overall"]:.4f}  '
              f'maj={r["HR_p2_major"]:.4f}  min={r["HR_p2_minor"]:.4f}  '
              f'({time.time()-t0:.0f}s)')

print('\nMulti-seed 학습 완료')

## 셀 7 — Aggregate + Sanity + 3-gate

Sanity: V3_ref P1 mean ≈ V3 reference P1 (0.6607) — Phase 1은 V3와 동일 protocol이므로 일치해야 함.
3-gate: C1 vs V3 reference (hardcoded EXP #15).

In [ ]:
def aggregate(cfg):
    rs = [all_results[(cfg, s)] for s in SEEDS]
    o = np.array([r['HR_p2_overall'] for r in rs])
    M = np.array([r['HR_p2_major'] for r in rs])
    m = np.array([r['HR_p2_minor'] for r in rs])
    p1 = np.array([r['HR_p1'] for r in rs])
    return dict(
        overall_mean=float(o.mean()), overall_std=float(o.std(ddof=1)),
        major_mean=float(M.mean()),   major_std=float(M.std(ddof=1)),
        minor_mean=float(m.mean()),   minor_std=float(m.std(ddof=1)),
        p1_mean=float(p1.mean()),     p1_std=float(p1.std(ddof=1)),
        per_seed_overall=o.tolist(),
    )

agg = {c: aggregate(c) for c in CONFIGS}

print(f'{"Config":<10} {"P1 mean±std":<18} {"P2 overall mean±std":<22} {"P2 major mean±std":<22} {"P2 minor mean±std":<22}')
print('-'*110)
for c in CONFIGS:
    a = agg[c]
    print(f'{c:<10} {a["p1_mean"]:.4f} ± {a["p1_std"]:.4f}  '
          f'{a["overall_mean"]:.4f} ± {a["overall_std"]:.4f}  '
          f'{a["major_mean"]:.4f} ± {a["major_std"]:.4f}  '
          f'{a["minor_mean"]:.4f} ± {a["minor_std"]:.4f}')

print(f'\nPer-seed P2 overall:')
for c in CONFIGS:
    vals = [all_results[(c, s)]['HR_p2_overall'] for s in SEEDS]
    print(f'  {c}: ' + '  '.join(f'seed={s} {h:.4f}' for s, h in zip(SEEDS, vals)))

# Sanity: V3_ref P1 should match EXP #15 V3 P1 (0.6607)
print(f'\n=== Sanity (V3_ref P1 ≈ EXP #15 V3 P1 {V3_REF_P1:.4f}) ===')
v3_ref_p1 = agg['V3_ref']['p1_mean']
sanity_p1 = abs(v3_ref_p1 - V3_REF_P1) <= 0.003
print(f'  V3_ref P1 = {v3_ref_p1:.4f}, EXP #15 P1 = {V3_REF_P1:.4f}, diff={v3_ref_p1-V3_REF_P1:+.4f}  {"O" if sanity_p1 else "X (protocol drift)"}')

# Sanity 2: V3_ref overall should match EXP #15 V3 overall (0.6615)
v3_ref_overall = agg['V3_ref']['overall_mean']
sanity_overall = abs(v3_ref_overall - V3_REF_OVERALL) <= 0.003
print(f'  V3_ref overall = {v3_ref_overall:.4f}, EXP #15 overall = {V3_REF_OVERALL:.4f}, diff={v3_ref_overall-V3_REF_OVERALL:+.4f}  {"O" if sanity_overall else "X (protocol drift)"}')

# 3-gate: 각 axis-wise config vs V3_ref (this run's control)
import math
n_test_configs = sum(1 for c in CONFIGS if c != 'V3_ref')  # 3
bonferroni = math.sqrt(n_test_configs)
v3r = agg['V3_ref']

print(f'\n=== 3-gate strict (각 config vs V3_ref, Bonferroni √{n_test_configs}) ===')
gate_results = {}
for cn in CONFIGS:
    if cn == 'V3_ref': continue
    a = agg[cn]
    d_o = a['overall_mean'] - v3r['overall_mean']
    d_M = a['major_mean']   - v3r['major_mean']
    d_m = a['minor_mean']   - v3r['minor_mean']
    combined_std = (a['overall_std']**2 + v3r['overall_std']**2) ** 0.5
    g1_thr = combined_std * bonferroni
    g1 = d_o > g1_thr
    g2 = d_M > -0.002
    g3 = d_m > -0.005
    pass3 = bool(g1 and g2 and g3 and sanity_p1 and sanity_overall)
    gate_results[cn] = dict(d_overall=float(d_o), d_major=float(d_M), d_minor=float(d_m),
                              thr=float(g1_thr), g1=bool(g1), g2=bool(g2), g3=bool(g3), pass3=pass3)
    sg = CONFIGS[cn]
    print(f'  {cn} σ=({sg[0]*1000:.1f},{sg[1]*1000:.1f},{sg[2]*1000:.1f})mm: '
          f'Δoverall={d_o:+.4f} (thr {g1_thr:+.4f})  Δmajor={d_M:+.4f}  Δminor={d_m:+.4f}  '
          f'G1={"O" if g1 else "X"} G2={"O" if g2 else "X"} G3={"O" if g3 else "X"}  '
          f'pass={"O" if pass3 else "X"}')

# 채택 결정: 통과 config 중 overall best
passed = [cn for cn in CONFIGS if cn != 'V3_ref' and gate_results[cn]['pass3']]
if not passed:
    chosen = None
    print(f'\n채택 없음 — 모든 axis-wise config가 V3_ref strict 우위 아님. lever 폐기 후보.')
elif len(passed) == 1:
    chosen = passed[0]
    print(f'\n채택: {chosen} (단일 통과)')
else:
    chosen = max(passed, key=lambda c: agg[c]['overall_mean'])
    print(f'\n채택: {chosen} (다중 통과 {passed} 중 overall best)')

# 추가 진단: V3_ref가 EXP #15 V3와 정합한지 (protocol drift 재확인)
print(f'\n=== Stage 1 결과 trend (lever direction 진단) ===')
for cn in CONFIGS:
    a = agg[cn]
    delta_vs_v3ref = a['overall_mean'] - v3r['overall_mean']
    sg = CONFIGS[cn]
    print(f'  {cn}: overall {a["overall_mean"]:.4f} (Δ vs V3_ref {delta_vs_v3ref:+.4f})  '
          f'σ=({sg[0]*1000:.1f},{sg[1]*1000:.1f},{sg[2]*1000:.1f})mm')

# Boundary 진단 (Stage 2 trigger)
if chosen in ('C2', 'C3'):
    print(f'\n⚠️ Boundary config 채택 ({chosen}) → Stage 2 σ 확장 검토 권장')
elif chosen == 'C1':
    print(f'\nInterior optimum 채택 (C1) → 1:1 std proportional 가설 확정')


## 셀 8 — Submission 생성 (양 config 모두 저장, 채택 config 우선)

In [ ]:
def make_submission(test_resid_avg, name):
    pred_test_abs = base_test + test_resid_avg
    assert pred_test_abs.shape == (10000, 3)
    assert np.isfinite(pred_test_abs).all()
    df = pd.DataFrame({
        'id': test_ids,
        'x': pred_test_abs[:, 0],
        'y': pred_test_abs[:, 1],
        'z': pred_test_abs[:, 2],
    })
    path = f'submission_t_ax_{name}.csv'
    df.to_csv(path, index=False)
    print(f'  {path}: x[{pred_test_abs[:,0].min():.2f},{pred_test_abs[:,0].max():.2f}] '
          f'y[{pred_test_abs[:,1].min():.2f},{pred_test_abs[:,1].max():.2f}] '
          f'z[{pred_test_abs[:,2].min():.2f},{pred_test_abs[:,2].max():.2f}]')
    return path

sub_paths = {}
for c in CONFIGS:
    avg = np.mean(test_preds[c], axis=0).astype(np.float32)
    p = make_submission(avg, f'{c}_ms3')
    sub_paths[c] = p

if chosen:
    print(f'\n채택 {chosen} → 우선 제출: {sub_paths[chosen]}')
else:
    print(f'\n채택 없음 — submission 생성됐으나 LB 제출 보류')

## 셀 9 — Meta 저장 + Drive 복사 + 로컬 다운로드

In [ ]:
def safe(x):
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.bool_): return bool(x)
    return x

agg_safe = {k: {kk: safe(vv) for kk, vv in v.items()} for k, v in agg.items()}

meta = {
    'protocol': 'T-AX (V3 base + axis-wise W1 σ on Frenet)',
    'seeds': SEEDS, 'n_folds': N_FOLDS,
    'configs': {k: list(v) for k, v in CONFIGS.items()},
    'max_w': BEST_MAX_W,
    'minority_threshold': MINORITY_THRESHOLD,
    'v3_reference_exp15': {'overall': V3_REF_OVERALL, 'overall_std': V3_REF_OVERALL_STD,
                            'major': V3_REF_MAJOR, 'minor': V3_REF_MINOR, 'p1': V3_REF_P1},
    'aggregate': agg_safe,
    'sanity_p1_pass': bool(sanity_p1),
    'sanity_overall_pass': bool(sanity_overall),
    'gate_results_vs_V3ref': {k: {kk: safe(vv) for kk, vv in v.items()} for k, v in gate_results.items()},
    'bonferroni_factor': float(bonferroni),
    'chosen': chosen,
    'submissions': sub_paths,
}
with open('results/t_ax_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('results/t_ax_meta.json 저장')

try:
    from google.colab import drive, files
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive'
    results_drive = os.path.join(DRIVE_BASE, 'results')
    os.makedirs(results_drive, exist_ok=True)
    !cp -r results/* {results_drive}/
    for p in sub_paths.values():
        !cp {p} {DRIVE_BASE}/
        files.download(p)
    print('Drive 복사 + 다운로드 완료')
except ImportError:
    print('Colab 환경 아님 — Drive 복사 skip')